# hybrid-kv-cliff: KV Eviction vs. Low-Bit Quantization in Sparse-Attention Hybrid LLMs
**Repository:** [`jaimejarauta/decoupled-delta-net`](https://github.com/jaimejarauta/decoupled-delta-net)
**Branch:** `feature/hybrid-kv-cliff`
**Hardware Default:** NVIDIA L4 GPU (24GB VRAM)
**Lead Researcher:** Jaime Jarauta
**One-Click Run:** Select **Runtime -> Run all**.


In [ ]:
# Step 1: Self-Healing Environment Setup, Bulletproof Credit Guard & Colab Secrets Activation
import os, sys, signal, subprocess
from IPython import get_ipython

def auto_teardown(signum=None, frame=None):
    print("\n[Credit Guard] Unhandled error, interrupt, or completion! Syncing W&B and unassigning GPU...")
    try:
        subprocess.run("wandb sync --include-offline 2>/dev/null || true", shell=True)
    except Exception:
        pass
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass

def custom_exc_handler(self, etype, value, tb, tb_offset=None):
    self.showtraceback((etype, value, tb), tb_offset=tb_offset)
    auto_teardown()
    return []

ip = get_ipython()
if ip is not None:
    ip.set_custom_exc((Exception,), custom_exc_handler)

# Activate API Secrets automatically from Colab Userdata / Secrets panel (Key icon 🔑)
try:
    from google.colab import userdata
    for k in ["HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "hf_token"]:
        try:
            tok = userdata.get(k)
            if tok:
                os.environ["HF_TOKEN"] = tok
                os.environ["HUGGING_FACE_HUB_TOKEN"] = tok
                print(f"[Secrets] Successfully loaded Hugging Face token from Colab Secrets ({k}).")
                break
        except Exception:
            pass
    for k in ["WANDB_API_KEY", "wandb_api_key", "WANDB_KEY"]:
        try:
            key = userdata.get(k)
            if key:
                os.environ["WANDB_API_KEY"] = key
                print(f"[Secrets] Successfully loaded Weights & Biases API key from Colab Secrets ({k}).")
                break
        except Exception:
            pass
except ImportError:
    pass

print("[Ready] Credit guard active. L4 GPU default configured. VM will self-terminate on error or completion.")


In [ ]:
# Step 2: Clone Repository (feature/hybrid-kv-cliff) & Mount Google Drive
import os, sys, subprocess
from pathlib import Path
import torch

assert torch.cuda.is_available(), "Switch Colab runtime to GPU (L4 24GB recommended): Runtime -> Change runtime type -> GPU (L4)"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB | BF16: {torch.cuda.is_bf16_supported()}")

repo_url = "https://github.com/jaimejarauta/decoupled-delta-net.git"
branch = "feature/hybrid-kv-cliff"
target_dir = "/content/hybrid-kv-cliff"

%cd /content
if not os.path.exists(target_dir):
    print(f"Cloning feature/hybrid-kv-cliff from {repo_url}...")
    subprocess.run(["git", "clone", "-b", branch, repo_url, target_dir], check=True)
else:
    print(f"Updating feature/hybrid-kv-cliff to latest...")
    subprocess.run(["git", "-C", target_dir, "fetch", "origin", branch], check=True)
    subprocess.run(["git", "-C", target_dir, "checkout", branch], check=True)
    subprocess.run(["git", "-C", target_dir, "reset", "--hard", f"origin/{branch}"], check=True)
    subprocess.run(["git", "-C", target_dir, "pull", "origin", branch], check=True)

%cd /content/hybrid-kv-cliff
!pip install -q -e .
!pip install -q "transformers>=5.0" "datasets>=3.0" accelerate pytest wandb scipy

# Auto-login to Hugging Face and W&B if secrets were loaded
if os.environ.get("HF_TOKEN"):
    try:
        import huggingface_hub
        huggingface_hub.login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
        print("[Auth] Hugging Face Hub successfully logged in.")
    except Exception as e:
        print(f"[Auth] HF login note: {e}")
if os.environ.get("WANDB_API_KEY"):
    try:
        import wandb
        wandb.login(key=os.environ["WANDB_API_KEY"])
        print("[Auth] Weights & Biases successfully logged in.")
    except Exception as e:
        print(f"[Auth] W&B login note: {e}")

# Zero-Prompt Storage: Save results to local runtime directory by default.
# (Set MOUNT_GOOGLE_DRIVE = True only if you explicitly wish to authenticate Google Drive).
MOUNT_GOOGLE_DRIVE = False
persist_dir = Path("/content/hybrid-kv-cliff/results")
persist_dir.mkdir(parents=True, exist_ok=True)

if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        persist_dir = Path("/content/drive/MyDrive/moonshot_results/hybrid-kv-cliff")
        persist_dir.mkdir(parents=True, exist_ok=True)
        print(f"[Google Drive] Persistent results directory mounted: {persist_dir}")
    except Exception as e:
        print(f"Drive notice: {e}")
else:
    print(f"[Zero-Prompt Run] Results stored locally at {persist_dir} (synced via W&B; no Drive popup)")

if "/content/hybrid-kv-cliff/src" not in sys.path:
    sys.path.insert(0, "/content/hybrid-kv-cliff/src")
os.environ["PYTHONPATH"] = "/content/hybrid-kv-cliff/src:" + os.environ.get("PYTHONPATH", "")
print("[Ready] hybrid-kv-cliff environment ready on GPU.")


In [ ]:
# Step 3: Fast CPU Unit Verification Tests
!pytest -q tests/test_compress.py


In [ ]:
# Step 4: Fast Smoke Pilot (Live Streaming)
print("=" * 70)
print("STEP 4: FAST SMOKE PILOT")
print("=" * 70)
!python scripts/run_kill.py run --model Qwen/Qwen3-1.7B-Base --lengths 8192 --n 10 --out {persist_dir}/smoke_pilot.jsonl
!python scripts/run_kill.py report --out {persist_dir}/smoke_pilot.jsonl


In [ ]:
# Step 5: Full Resumable Pre-Registered Kill Sweep (Live Streaming)
print("=" * 70)
print("STEP 5: FULL RESUMABLE PRE-REGISTERED KILL SWEEP")
print("=" * 70)
!python scripts/run_kill.py run --model Qwen/Qwen3.5-2B-Base --lengths 8192 16384 --n 50 --out {persist_dir}/kill.jsonl
!python scripts/run_kill.py report --out {persist_dir}/kill.jsonl


In [ ]:
# Step 6: Sync Telemetry & Unassign GPU (Zero Credit Leakage)
print("[Credit Guard] Sweep complete! Syncing W&B and unassigning GPU...")
try:
    auto_teardown()
except NameError:
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        print(f"Teardown note: {e}")
